# Q-Agent Performance Analysis and Comparison

This notebook evaluates the newly developed **Dueling Double DQN Agent (`q-agent`)** against three baseline agents (`random`, `hunt`, and `bayes`) across all 10 fleet placement methods (including the 9 original methods and the new cognitive psychology-inspired `cognitive_human` method).

### Evaluation Setup
- We run a headless sweep of 200 games for each agent-placement configuration (4 agents × 10 placement methods = 40 configurations, total 8,000 games).
- The performance metric is **Average Turns** to complete a game (win/loss is also tracked).
- We load the advanced Dueling DDQN model checkpoint from `checkpoints/q_test.pt`.

In [ ]:
import logging
from pathlib import Path
import altair as alt
import polars as pl
from game.fleet_placement_methods import PLACEMENT_METHODS
from game.game_logger import GameLogger
from main import run_games_headless

GameLogger.setup(console_level=logging.WARNING)
alt.data_transformers.enable("vegafusion")

In [ ]:
# Configuration
GAMES_PER_CONFIG = 200
AGENT_TYPES = ["random", "hunt", "bayes", "biased-bayes", "q-agent", "cognitive-q"]
PLACEMENT_METHODS_LIST = list(PLACEMENT_METHODS.keys())
Q_CHECKPOINT = "checkpoints/q_test.pt"
OUTPUT_CSV = Path("data/q_agent_comparison.csv")
OUTPUT_IMG_DIR = Path("img")
OUTPUT_IMG_DIR.mkdir(exist_ok=True)

print(f"Sweep Config:")
print(f"- Agents: {AGENT_TYPES}")
print(f"- Placement Methods: {PLACEMENT_METHODS_LIST}")
print(f"- Games per config: {GAMES_PER_CONFIG}")

In [ ]:
# Run evaluations across all configurations
records = []
configs = [(a, m) for a in AGENT_TYPES for m in PLACEMENT_METHODS_LIST]
n = len(configs)

for i, (agent_type, method) in enumerate(configs):
    print(f"[{i + 1}/{n}] Evaluating {agent_type} against {method} placement...")
    
    # Load the correct checkpoint for each Q-agent variant
    if agent_type == "cognitive-q":
        ckpt = "checkpoints/q_cognitive.pt"
    elif agent_type == "q-agent":
        ckpt = Q_CHECKPOINT
    else:
        ckpt = "checkpoints/q_agent.pt"
    
    games = run_games_headless(
        agent_type=agent_type,
        player_placement_method=method,
        n_games=GAMES_PER_CONFIG,
        checkpoint_path=ckpt
    )
    
    for game_id, game in enumerate(games):
        records.append({
            "agent_type": agent_type,
            "placement_method": method,
            "game_id": game_id,
            **game
        })

df = pl.DataFrame(records)
df.write_csv(OUTPUT_CSV)
print(f"\nEvaluation complete! Saved {len(df):,} game results to {OUTPUT_CSV}")

In [ ]:
# Aggregate results and compute metrics
summary = (
    df.group_by(["agent_type", "placement_method"])
    .agg(
        pl.col("agent_won").mean().alias("win_rate"),
        pl.col("turns").mean().alias("avg_turns"),
        pl.col("agent_hits").mean().alias("avg_agent_hits"),
        pl.col("player_hits").mean().alias("avg_player_hits"),
        pl.col("agent_sunk").mean().alias("avg_agent_sunk"),
        pl.col("player_sunk").mean().alias("avg_player_sunk"),
        pl.col("game_id").count().alias("n_games"),
    )
    .with_columns(
        (pl.col("win_rate") * 100).round(1).alias("win_rate_pct"),
        pl.col("avg_turns").round(1),
    )
    .sort(["agent_type", "placement_method"])
)
print("Aggregated Performance Summary:")
summary

In [ ]:
# Define and render comparison heatmap visualization
def chart_title(text: str) -> alt.TitleParams:
    return alt.TitleParams(text, fontSize=14, fontWeight="bold", anchor="middle")

def make_heatmap(df: pl.DataFrame, title: str) -> alt.LayerChart:
    heatmap = (
        alt.Chart(df)
        .mark_rect()
        .encode(
            x=alt.X(
                "placement_method:N",
                title="Player Placement Method",
                axis=alt.Axis(labelAngle=-35),
            ),
            y=alt.Y("agent_type:N", title="Agent Type"),
            color=alt.Color(
                "avg_turns:Q",
                title="Average Turns",
                scale=alt.Scale(scheme="blues", domain=[100, 30]),
            ),
            tooltip=[
                alt.Tooltip("agent_type:N", title="Agent"),
                alt.Tooltip("placement_method:N", title="Placement"),
                alt.Tooltip("win_rate_pct:Q", title="Win Rate (%)", format=".1f"),
                alt.Tooltip("avg_turns:Q", title="Avg Turns", format=".1f"),
            ],
        )
        .properties(
            title=chart_title(title),
            width=600,
            height=200,
        )
    )

    labels = (
        alt.Chart(df)
        .mark_text(fontSize=11, fontWeight="bold")
        .encode(
            x=alt.X("placement_method:N"),
            y=alt.Y("agent_type:N"),
            text=alt.Text("avg_turns:Q", format=".0f"),
            color=alt.condition(
                alt.datum.avg_turns < 60,
                alt.value("white"),
                alt.value("black"),
            ),
        )
    )

    return heatmap + labels

c = make_heatmap(summary, "Agent Performance by Type and Player Placement Method")
# Export visual artifact
c.save(str(OUTPUT_IMG_DIR / "agent_comparison_heatmap.png"))
c

## Q-Agent - Additional Analysis (Multi-Panel Visualization)

Here we load the test data `data/q_agent_test.csv` (10,000 games) and plot three custom analysis panels:
1. **Turns per Game Distribution** (Histogram)
2. **Agent Turns: Wins vs Losses** (Strip plot with mean indicator ticks)
3. **Hit Accuracy** (100-game rolling average of agent vs player accuracy)

In [ ]:
# Load test results for detailed analysis
test_df = pl.read_csv("data/q_agent_test.csv")

# 1. Turns per Game Distribution (Histogram)
hist = (
    alt.Chart(alt.Data(values=test_df.to_dicts()))
    .mark_bar(color="#4682b4")
    .encode(
        x=alt.X("turns:Q", bin=alt.Bin(step=5), title="Total Turns"),
        y=alt.Y("count():Q", title="Games"),
    )
    .properties(
        title="Turns per Game Distribution",
        width=220,
        height=220
    )
)

# 2. Agent Turns: Wins vs Losses (Strip plot)
test_df_status = test_df.with_columns([
    pl.when(pl.col("agent_won") == True)
    .then(pl.lit("Agent Win"))
    .otherwise(pl.lit("Agent Loss"))
    .alias("status")
])

strip = (
    alt.Chart(alt.Data(values=test_df_status.to_dicts()))
    .mark_circle(opacity=0.3, size=15, color="#ff7f0e")
    .encode(
        x=alt.X("status:N", title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("turns:Q", title="Total Turns", scale=alt.Scale(domain=[0, 80])),
    )
    .properties(
        title="Agent Turns: Wins vs Losses",
        width=220,
        height=220
    )
)

mean_ticks = (
    alt.Chart(alt.Data(values=test_df_status.to_dicts()))
    .mark_tick(color="black", thickness=3, size=40)
    .encode(
        x="status:N",
        y="mean(turns):Q"
    )
)
strip_chart = strip + mean_ticks

# 3. Hit Accuracy (Dual Rolling Line chart)
test_df_acc = test_df.with_columns([
    (pl.col("agent_hit") / pl.col("turns") * 100).alias("agent_acc"),
    (pl.col("player_hit") / pl.col("turns") * 100).alias("player_acc"),
])

test_df_acc = test_df_acc.with_columns([
    pl.col("agent_acc").rolling_mean(window_size=100, min_samples=1).alias("agent_acc_roll"),
    pl.col("player_acc").rolling_mean(window_size=100, min_samples=1).alias("player_acc_roll"),
])

df_long = test_df_acc.select([
    pl.col("game"),
    pl.col("agent_acc_roll").alias("Agent"),
    pl.col("player_acc_roll").alias("Player"),
]).unpivot(
    index="game",
    on=["Agent", "Player"],
    variable_name="Type",
    value_name="Accuracy"
)

accuracy_chart = (
    alt.Chart(alt.Data(values=df_long.to_dicts()))
    .mark_line(strokeWidth=1.5, opacity=0.8)
    .encode(
        x=alt.X("game:Q", title="Game", scale=alt.Scale(domain=[0, 10000])),
        y=alt.Y("Accuracy:Q", title="Hit Accuracy (%) (100-game rolling avg)", scale=alt.Scale(domain=[0, 45])),
        color=alt.Color("Type:N", title=None, scale=alt.Scale(range=["#1f77b4", "#ff7f0e"]))
    )
    .properties(
        title="Hit Accuracy (hits/turns %)",
        width=220,
        height=220
    )
)

# Concat and export Panel 1
additional_analysis = alt.hconcat(hist, strip_chart, accuracy_chart).properties(
    title=alt.TitleParams(
        "Q-Agent - Additional Analysis",
        fontSize=15,
        fontWeight="bold",
        anchor="middle"
    )
)
additional_analysis.save(str(OUTPUT_IMG_DIR / "q_agent_additional_analysis.png"))
additional_analysis

## Q-Learning Training - Mean Turns (Convergence History)

This panel visualizes the DQN model's learning progression over the first 3.4k episodes, illustrating rolling average turns per episode along with several standard target baseline rules (`bayes` and `hunt`) and best training metrics.

In [ ]:
# Load training progression data
train_df = pl.read_csv("data/q_agent_train.csv")
train_df_filtered = train_df.filter(pl.col("episode") <= 3400)

best_raw_val = train_df_filtered["mean_turns"].min()
train_df_filtered = train_df_filtered.with_columns([
    pl.col("mean_turns").rolling_mean(window_size=5, min_samples=1).alias("rolling_mean_turns")
])
best_rolling_val = train_df_filtered["rolling_mean_turns"].min()

# Prepare long-format data structure for unified legend in Altair
main_line_data = []
for row in train_df_filtered.to_dicts():
    main_line_data.append({
        "episode": row["episode"],
        "value": row["mean_turns"],
        "Type": "Rolling avg"
    })

for ep in [0, 3400]:
    main_line_data.append({"episode": ep, "value": best_raw_val, "Type": "Best raw"})
    main_line_data.append({"episode": ep, "value": best_rolling_val, "Type": "Best rolling"})
    main_line_data.append({"episode": ep, "value": 47.1, "Type": "Bayes target"})
    main_line_data.append({"episode": ep, "value": 55.3, "Type": "Hunt target"})

train_chart = alt.Chart(alt.Data(values=main_line_data)).mark_line().encode(
    x=alt.X(
        "episode:Q",
        title="Episode",
        axis=alt.Axis(
            values=list(range(200, 3600, 200)),
            labelExpr="datum.value / 1000 + 'k'"
        )
    ),
    y=alt.Y("value:Q", title="Mean turns (1-episode rolling avg)", scale=alt.Scale(domain=[0, 70])),
    color=alt.Color("Type:N", title=None, scale=alt.Scale(
        domain=["Rolling avg", "Best rolling", "Best raw", "Bayes target", "Hunt target"],
        range=["#1f77b4", "grey", "red", "#2c3e50", "#7f8c8d"]
    )),
    strokeDash=alt.StrokeDash("Type:N", title=None, scale=alt.Scale(
        domain=["Rolling avg", "Best rolling", "Best raw", "Bayes target", "Hunt target"],
        range=[[], [2, 2], [4, 4], [4, 4], [4, 4]]
    ))
).properties(
    title="Q-Learning Training - Mean Turns",
    width=500,
    height=300
)

train_chart.save(str(OUTPUT_IMG_DIR / "q_agent_training_convergence.png"))
train_chart